In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

print("Loading data...")
df = pd.read_csv('../data/processed/fake_jobs_clean.csv')
print(f"Loaded: {len(df)} rows")

# Combine ALL text columns
df['text'] = (df['title'].fillna('') + ' ' + 
              df['description'].fillna('') + ' ' + 
              df['requirements'].fillna('') + ' ' +
              df['company_profile'].fillna('') + ' ' +
              df['benefits'].fillna(''))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    return ' '.join(text.split())

print("Cleaning text...")
df['text_clean'] = df['text'].apply(clean_text)

# TF-IDF with 2000 features + bigrams
print("Creating TF-IDF features (this will take 5-10 minutes)...")
tfidf = TfidfVectorizer(
    max_features=2000,
    stop_words='english',
    ngram_range=(1, 2),  # unigrams + bigrams
    min_df=2,  # ignore words that appear in only 1 doc
    max_df=0.8  # ignore words that appear in >80% of docs
)
X_tfidf = tfidf.fit_transform(df['text_clean'])
print(f"TF-IDF done: {X_tfidf.shape}")

# Richer numeric features
df['text_length'] = df['text_clean'].str.len()
df['has_salary'] = df['salary_range'].notna().astype(int)
df['has_company_logo'] = df['has_company_logo'].astype(int)
df['has_questions'] = df['has_questions'].astype(int)
df['has_department'] = df['department'].notna().astype(int)
df['has_benefits'] = df['benefits'].notna().astype(int)
df['title_length'] = df['title'].fillna('').str.len()
df['desc_length'] = df['description'].fillna('').str.len()

numeric_cols = ['text_length', 'has_salary', 'has_company_logo', 'has_questions',
                'has_department', 'has_benefits', 'title_length', 'desc_length']

from scipy.sparse import hstack
X = hstack([X_tfidf, df[numeric_cols].values])
y = df['fraudulent'].values

print(f"Final feature matrix: {X.shape}")
print(f"Fake jobs: {y.sum()}, Real jobs: {len(y) - y.sum()}")

# Save
os.makedirs('../src/features', exist_ok=True)
with open('../src/features/tfidf_vectorizer_v2.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
with open('../src/features/X_y_v2.pkl', 'wb') as f:
    pickle.dump((X, y), f)

print("Saved to src/features/")

Loading data...
Loaded: 17880 rows
Cleaning text...
Creating TF-IDF features (this will take 5-10 minutes)...
TF-IDF done: (17880, 2000)
Final feature matrix: (17880, 2008)
Fake jobs: 866, Real jobs: 17014
Saved to src/features/
